# Questões para discussão

No relatório e na apresentação, responda:

1. Como o problema foi modelado como grafo?
- O problema foi modelado como um Grafo Direcionado (Dígrafo) utilizando a biblioteca OSMnx. A malha viária real de Natal foi capturada e convertida em uma estrutura de dados de lista/dicionário de adjacência. Essa estrutura permite representar de forma fiel as regras de trânsito reais, como ruas de mão única, conversões proibidas e retornos.
2. O que representam os nós e as arestas?
- Nós (Vértices): Representam os cruzamentos de vias, bifurcações, esquinas e pontos geométricos de conexão na malha urbana.
- Arestas (Arcos): Representam os segmentos de ruas e avenidas que conectam esses cruzamentos, possuindo um sentido de direção obrigatório.
3. Quais pesos foram usados?
- Distância (grafo_distancia): O comprimento físico da via medido em metros.
- Tempo Livre (grafo_tempo): O tempo teórico (em segundos) gasto para percorrer a via baseando-se no limite máximo de velocidade regulamentar.
- Tempo com Trânsito Realista (grafo_transito): O tempo de viagem calibrado de acordo com uma velocidade média de horário de pico urbano ($20\text{ km/h}$), garantindo sincronia com ferramentas comerciais como o Google Maps.
4. Como o trânsito sintético alterou as rotas?
- A inclusão do trânsito mudou drasticamente o comportamento das rotas. No cenário de tempo livre, o algoritmo tende a escolher caminhos em linha reta por vias locais. Com o trânsito pesado ativado, o algoritmo de caminho mínimo passa a penalizar severamente vias saturadas e redireciona o veículo para rotas que priorizam o fluxo contínuo (como grandes avenidas ou rodovias periféricas), mesmo que o trajeto físico se torne geograficamente mais longo.
5. Caminhar alguns metros melhorou a solução?
- Sim. No experimento prático de Ponta Negra ao Midway Mall, caminhar apenas $61\text{ metros}$ permitiu ao usuário se deslocar até um nó estratégico fora do gargalo local de origem. Isso permitiu que o veículo de aplicativo fizesse o embarque em uma posição muito mais favorável, economizando minutos preciosos que seriam gastos em retornos residenciais lentos ou semáforos de bairros.
6. Em quais casos caminhar atrapalhou?
- Caminhar atrapalha a solução em duas situações principais: Quando o raio máximo de caminhada ($X$) é excessivamente grande: O algoritmo pode sugerir que o usuário caminhe distâncias irreais (ex: mais de $1\text{ km}$ a pé) sob sol ou chuva para poupar uma diferença insignificante de tempo de carro, reduzindo o conforto do serviço de mobilidade. E Origens em vias de fluxo livre: Se o ponto de partida original $A$ já estiver localizado em uma avenida de trânsito rápido e sentido favorável, qualquer caminhada para nós adjacentes apenas adicionará cansaço físico ao usuário sem gerar ganho no trecho de carro.
7. A menor distância foi também a rota mais rápida?
- Não. O teste de validação comprovou que a rota baseada estritamente em menor distância física (metros) força o trajeto por dentro de bairros com muitos cruzamentos e conversões de baixa velocidade. Já a rota baseada em tempo com trânsito priorizou o deslocamento por vias arteriais de maior fluidez, provando que, no cenário urbano, o caminho geometricamente mais curto raramente é o mais rápido.
8. O A* expandiu menos nós que o Dijkstra?
- Sim, drasticamente menos. Enquanto o Dijkstra expande nós de maneira radial e concêntrica (em círculos para todas as direções a partir da origem), o $A^*$ utiliza uma função heurística baseada na distância em linha reta até o destino. Isso dá um "senso de direção" ao algoritmo, fazendo com que ele expanda nós focando quase que exclusivamente na direção geométrica do Midway Mall, reduzindo o espaço de busca e o tempo de processamento.
9. O Dijkstra com Heap foi mais eficiente que o Dijkstra simples?
- Sim, em termos de complexidade de tempo. O Dijkstra simples varre linearmente ($O(V)$) todo o conjunto de nós a cada iteração para descobrir qual possui a menor distância atual. O Dijkstra com Heap utiliza uma Fila de Prioridade (Min-Heap), reduzindo esse custo de busca do menor elemento para $O(\log V)$. Embora o número de nós expandidos no pior caso possa ser similar, o tempo de CPU gasto pela estrutura com Heap é imensamente menor em grafos grandes.
10. O algoritmo da literatura trouxe algum ganho?
- O algoritmo adicional implementado (Dijkstra Bidirecional) trouxe um excelente ganho computacional. Ao disparar duas buscas simultâneas — uma direto da origem e outra reversa do destino — as franjas de busca se encontram no meio do caminho. Isso faz com que a área total de nós explorados seja consideravelmente menor do que uma busca tradicional do Dijkstra que precisa caminhar sozinha por todo o trajeto.
11. Quais limitações existem na modelagem proposta?
- Independência de Arestas: Na primeira versão do trânsito sintético (random.uniform), o trânsito flutuava quarteirão por quarteirão. Na realidade, o trânsito é regionalmente correlacionado (se a Engenheiro Roberto Freire para, todos os quarteirões dela travam em cadeia).
- Custo da Caminhada Abstrato: O modelo foca apenas em minimizar o tempo que o usuário passa dentro do carro, sem ponderar o "peso psicológico" ou o tempo gasto no deslocamento a pé pelo usuário até o ponto $P$.
- Grafo Estático: O mapa não sofre atualizações de eventos dinâmicos em tempo real, como acidentes repentinos, obras na pista ou bloqueios temporários.
12. Como o modelo poderia ser aproximado de um aplicativo real de mobilidade?
- Para aproximar este projeto de um sistema comercial como o Uber ou Waze, deveriam ser implementadas as seguintes melhorias:
- Função de Custo Bi-Objetivo: O algoritmo deveria otimizar uma função combinada, ponderando o tempo de carro e o tempo de caminhada multiplicados por fatores de conveniência (ex: 1 minuto andando "custa" o dobro do que 1 minuto confortavelmente sentado no carro).
- Fatores de Trânsito Dinâmicos e Coletivos: Substituir as penalidades estáticas ou puramente aleatórias por matrizes de trânsito baseadas em horários históricos (matriz origem-destino por faixa de horário) e velocidade média dos próprios usuários trafegando na via.
- Restrições de Segurança no Embarque: Adicionar uma camada de filtragem que impeça o ponto de embarque $P$ de ser alocado em locais perigosos, vias de alta velocidade sem acostamento (ex: viadutos ou pontes) ou locais onde a parada de veículos seja proibida por lei.



In [1]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.8 MB/s eta 0:00:00


In [2]:
import osmnx as ox
import networkx as nx
G_osm = ox.graph_from_place("Natal, Rio Grande do Norte, Brazil", network_type='drive')
G_osm = ox.add_edge_speeds(G_osm)
G_osm = ox.add_edge_travel_times(G_osm)
G = G_osm
coordenadas = {no: (G_osm.nodes[no]['y'], G_osm.nodes[no]['x']) for no in G_osm.nodes()}

In [3]:
def dijkstra_simples(grafo, origem, destino):
    distancias = {no: float('inf') for no in grafo}
    distancias[origem] = 0
    visitados = set()
    caminho_anterior = {no: None for no in grafo}
    nos_expandidos = 0

    while len(visitados) < len(grafo):
        # Encontra o noh não visitado com a menor distância atual
        no_atual = None
        menor_distancia = float('inf')
        for no in grafo:
            if no not in visitados and distancias[no] < menor_distancia:
                menor_distancia = distancias[no]
                no_atual = no

        # Se não encontrou um noh alcançável ou chegou ao destino, para
        if no_atual is None or no_atual == destino:
            break

        visitados.add(no_atual)
        nos_expandidos += 1

        # Relaxamento das arestas
        for vizinho, peso in grafo[no_atual].items():
            nova_distancia = distancias[no_atual] + peso
            if nova_distancia < distancias[vizinho]:
                distancias[vizinho] = nova_distancia
                caminho_anterior[vizinho] = no_atual

    # Reconstrói o caminho
    caminho = []
    atual = destino
    while atual is not None:
        caminho.insert(0, atual)
        atual = caminho_anterior[atual]

    if distancias[destino] == float('inf'):
        return None, float('inf'), nos_expandidos # Caminho não encontrado

    return caminho, distancias[destino], nos_expandidos

In [4]:
import heapq

def dijkstra_heap(grafo, origem, destino):
    distancias = {no: float('inf') for no in grafo}
    distancias[origem] = 0
    # A fila armazena tuplas: (distancia_acumulada, no_atual)
    fila_prioridade = [(0, origem)]
    caminho_anterior = {no: None for no in grafo}
    nos_expandidos = 0

    while fila_prioridade:
        distancia_atual, no_atual = heapq.heappop(fila_prioridade)

        # Se já encontramos um caminho mais curto antes, ignoramos
        if distancia_atual > distancias[no_atual]:
            continue

        nos_expandidos += 1

        if no_atual == destino:
            break

        # Relaxamento das arestas
        for vizinho, peso in grafo[no_atual].items():
            nova_distancia = distancia_atual + peso

            if nova_distancia < distancias[vizinho]:
                distancias[vizinho] = nova_distancia
                caminho_anterior[vizinho] = no_atual
                heapq.heappush(fila_prioridade, (nova_distancia, vizinho))

    # Reconstrói o caminho
    caminho = []
    atual = destino
    while atual is not None:
        caminho.insert(0, atual)
        atual = caminho_anterior[atual]

    if distancias[destino] == float('inf'):
        return None, float('inf'), nos_expandidos # Caminho não encontrado

    return caminho, distancias[destino], nos_expandidos

A*

In [5]:
import math
def heuristica_geografica(grafo, no_atual, destino, coordenadas=None, velocidade_kmh=40):
    if coordenadas and no_atual in coordenadas and destino in coordenadas:
        lat1, lon1 = coordenadas[no_atual]
        lat2, lon2 = coordenadas[destino]

        distancia_metros = math.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2) * 111000

        velocidade_ms = velocidade_kmh / 3.6

        return distancia_metros / velocidade_ms
    return 0

def a_estrela(grafo, origem, destino, coordenadas=None):
    distancias = {no: float('inf') for no in grafo}
    distancias[origem] = 0


    h_inicial = heuristica_geografica(grafo, origem, destino, coordenadas)
    fila_prioridade = [(0 + h_inicial, 0, origem)]

    caminho_anterior = {no: None for no in grafo}
    nos_expandidos = 0
    visitados = set()

    while fila_prioridade:
        f_atual, g_atual, no_atual = heapq.heappop(fila_prioridade)

        if no_atual in visitados:
            continue

        visitados.add(no_atual)
        nos_expandidos += 1

        if no_atual == destino:
            break

        for vizinho, peso in grafo[no_atual].items():
            novo_g = g_atual + peso

            if novo_g < distancias[vizinho]:
                distancias[vizinho] = novo_g
                caminho_anterior[vizinho] = no_atual
                f_score = novo_g + heuristica_geografica(grafo, vizinho, destino, coordenadas)
                heapq.heappush(fila_prioridade, (f_score, novo_g, vizinho))

    caminho = []
    atual = destino
    while atual is not None:
        caminho.insert(0, atual)
        atual = caminho_anterior[atual]

    if distancias[destino] == float('inf'):
        return None, float('inf'), nos_expandidos

    return caminho, distancias[destino], nos_expandidos

Dijkstra Bidirecional

In [6]:
import heapq

def DijkstraBidirecional(grafo, grafo_reverso, origem, destino):
    if origem == destino:
        return [origem], 0

    dist_f = {no: float('inf') for no in grafo}
    dist_b = {no: float('inf') for no in grafo}

    pai_f = {origem: None}
    pai_b = {destino: None}

    dist_f[origem] = 0
    dist_b[destino] = 0

    pq_f = [(0, origem)]
    pq_b = [(0, destino)]

    processado_f = set()
    processado_b = set()

    mu = float('inf')
    no_encontrado = None

    while pq_f and pq_b:
        if pq_f[0][0] + pq_b[0][0] >= mu:
            break
        d_f, u = heapq.heappop(pq_f)
        if u in processado_f:
            continue
        processado_f.add(u)

        for v, peso in grafo.get(u, []):
            nova_dist = dist_f[u] + peso
            if nova_dist < dist_f.get(v, float('inf')):
                dist_f[v] = nova_dist
                pai_f[v] = u
                heapq.heappush(pq_f, (nova_dist, v))
                if v in processado_b:
                    dist_total = dist_f[v] + dist_b[v]
                    if dist_total < mu:
                        mu = dist_total
                        no_encontrado = v

        d_b, u_rev = heapq.heappop(pq_b)
        if u_rev in processado_b:
            continue
        processado_b.add(u_rev)

        for w, peso in grafo_reverso.get(u_rev, []):
            nova_dist = dist_b[u_rev] + peso
            if nova_dist < dist_b.get(w, float('inf')):
                dist_b[w] = nova_dist
                pai_b[w] = u_rev
                heapq.heappush(pq_b, (nova_dist, w))
                if w in processado_f:
                    dist_total = dist_f[w] + dist_b[w]
                    if dist_total < mu:
                        mu = dist_total
                        no_encontrado = w

    if mu == float('inf'):
        return None, float('inf')

    caminho = []

    atual = no_encontrado
    while atual is not None:
        caminho.append(atual)
        atual = pai_f.get(atual)
    caminho.reverse()

    atual = pai_b.get(no_encontrado)
    while atual is not None:
        caminho.append(atual)
        atual = pai_b.get(atual)

    return caminho, mu

In [7]:
# 2. CONSTRUINDO OS GRAFOS COMO DICIONÁRIOS {vizinho: peso}
print("Construindo os grafos de distância, tempo livre e tempo com trânsito...")

grafo_dist      = {no: {} for no in G.nodes()}
grafo_rev_dist  = {no: {} for no in G.nodes()}
grafo_tempo     = {no: {} for no in G.nodes()}
grafo_rev_tempo = {no: {} for no in G.nodes()}
grafo_tempo_livre     = {no: {} for no in G.nodes()}
grafo_rev_tempo_livre = {no: {} for no in G.nodes()}

PENALIDADE_CRUZAMENTO = 15  # segundos adicionados por aresta (trânsito sintético base)

for u, v, dados in G.edges(data=True):
    # --- MÉTRICA 1: DISTÂNCIA FÍSICA ---
    distancia = dados.get('length', 1.0)
    if v not in grafo_dist[u] or distancia < grafo_dist[u][v]:
        grafo_dist[u][v]     = distancia
        grafo_rev_dist[v][u] = distancia

    # --- MÉTRICA 2: TEMPO SEM TRÂNSITO (TEMPO LIVRE) ---
    # Usa apenas o travel_time base original da via
    tempo_base = dados.get('travel_time', 1.0)
    if v not in grafo_tempo_livre[u] or tempo_base < grafo_tempo_livre[u][v]:
        grafo_tempo_livre[u][v]     = tempo_base
        grafo_rev_tempo_livre[v][u] = tempo_base

    # --- MÉTRICA 3: TEMPO COM TRÂNSITO SINTÉTICO ---
    peso_tempo = tempo_base + PENALIDADE_CRUZAMENTO

    tipo_via = dados.get('highway', '')
    if isinstance(tipo_via, list):
        tipo_via = tipo_via[0]

    if tipo_via in ['primary', 'secondary', 'trunk']:
        peso_tempo *= 1.6
    elif tipo_via == 'residential':
        peso_tempo *= 1.2

    if v not in grafo_tempo[u] or peso_tempo < grafo_tempo[u][v]:
        grafo_tempo[u][v]     = peso_tempo
        grafo_rev_tempo[v][u] = peso_tempo

print("✅ Grafos de distância, tempo livre e tempo com trânsito construídos com sucesso.")
print(f"   Nós: {len(G.nodes())} | Arestas: {len(G.edges())}")

Construindo os grafos de distância, tempo livre e tempo com trânsito...
✅ Grafos de distância, tempo livre e tempo com trânsito construídos com sucesso.
   Nós: 18578 | Arestas: 48183


# RideSmart — Definição de Origem, Destino e Raio de Caminhada

Preencha os campos abaixo com o endereço de origem, o endereço de destino e a distância
máxima (em metros) que você está disposto a caminhar até o ponto de embarque.


In [8]:
# ============================================================
# CONFIGURAÇÃO DO RIDESMART — EDITE OS VALORES ABAIXO
# ============================================================

endereco_origem  = input("Endereço de ORIGEM  (ex: Midway Mall, Natal, Brazil): ").strip()
endereco_destino = input("Endereço de DESTINO (ex: Partage Norte Shopping, Natal, Brazil): ").strip()
raio_caminhada   = float(input("Distância máxima de CAMINHADA em metros (ex: 500): ").strip())

# Geocodifica e encontra os nós no grafo
print("\nBuscando coordenadas...")
coord_A = ox.geocode(endereco_origem)
coord_B = ox.geocode(endereco_destino)

no_A = ox.distance.nearest_nodes(G_osm, X=coord_A[1], Y=coord_A[0])
no_B = ox.distance.nearest_nodes(G_osm, X=coord_B[1], Y=coord_B[0])

print(f"✅ Origem  : {endereco_origem}")
print(f"   → nó OSM: {no_A}  (lat={coord_A[0]:.5f}, lon={coord_A[1]:.5f})")
print(f"✅ Destino : {endereco_destino}")
print(f"   → nó OSM: {no_B}  (lat={coord_B[0]:.5f}, lon={coord_B[1]:.5f})")
print(f"✅ Raio máx de caminhada: {raio_caminhada:.0f} m")


Endereço de ORIGEM  (ex: Midway Mall, Natal, Brazil): Midway Mall, Natal, Brazil
Endereço de DESTINO (ex: Partage Norte Shopping, Natal, Brazil): Ponta Negra, Natal, Brazil
Distância máxima de CAMINHADA em metros (ex: 500): 150

Buscando coordenadas...
✅ Origem  : Midway Mall, Natal, Brazil
   → nó OSM: 501969167  (lat=-5.81212, lon=-35.20572)
✅ Destino : Ponta Negra, Natal, Brazil
   → nó OSM: 7217605287  (lat=-5.87360, lon=-35.17663)
✅ Raio máx de caminhada: 150 m


In [9]:
import osmnx as ox
import folium

# ============================================================
# FUNÇÃO PRINCIPAL DO RIDESMART
# ============================================================

def selecionar_melhor_embarque(G_osm, grafo_carro, grafo_dist_dict,
                                origem_A, destino_B, dist_max_caminhada,
                                algoritmo_busca, coordenadas,
                                otimizar_por='tempo'):
    """
    Encontra o melhor ponto de embarque P dentro do raio de caminhada.

    Parâmetros
    ----------
    G_osm            : grafo OSMnx original (para BFS de vizinhança)
    grafo_carro      : grafo dict{no: dict{vizinho: peso}} com pesos de carro
                       (distância OU tempo, dependendo de otimizar_por)
    grafo_dist_dict  : grafo dict{no: dict{vizinho: distancia}} para medir
                       a caminhada real em metros
    origem_A         : nó de origem
    destino_B        : nó de destino
    dist_max_caminhada: raio máximo de caminhada em metros
    algoritmo_busca  : função (grafo, origem, destino, **kw) -> (caminho, custo, nos_exp)
                       Aceita dijkstra_simples, dijkstra_heap ou a_estrela.
    coordenadas      : dict{no: (lat, lon)} — usado pelo A*
    otimizar_por     : 'tempo' ou 'distancia'

    Retorna
    -------
    melhor_P, dist_caminhada, custo_carro, caminho_carro, nos_expandidos_medio
    """
    # Candidatos a ponto de embarque: nós dentro do raio por BFS de distância
    candidatos = ox.distance.nearest_nodes(
        G_osm,
        X=[G_osm.nodes[no]['x'] for no in G_osm.nodes()],
        Y=[G_osm.nodes[no]['y'] for no in G_osm.nodes()]
    ) if False else None   # placeholder — usamos a abordagem abaixo

    # Coleta todos os nós cuja distância geodésica de A é <= raio
    # (ox.distance.nearest_nodes não serve aqui; usamos ego_graph)
    subgrafo = nx.ego_graph(G_osm, origem_A, radius=dist_max_caminhada, distance='length')
    candidatos = list(subgrafo.nodes())

    melhor_P          = None
    melhor_custo      = float('inf')
    melhor_caminho    = None
    melhor_dist_cam   = 0
    total_nos_exp     = 0
    buscas_realizadas = 0

    for P in candidatos:
        if P == destino_B:
            continue

        # Distância real de caminhada A → P (metros)
        try:
            dist_cam = nx.shortest_path_length(G_osm, origem_A, P, weight='length')
        except nx.NetworkXNoPath:
            continue

        if dist_cam > dist_max_caminhada:
            continue

        # Rota de carro P → B com o algoritmo escolhido
        if algoritmo_busca == a_estrela:
            resultado = algoritmo_busca(grafo_carro, P, destino_B, coordenadas=coordenadas)
        else:
            resultado = algoritmo_busca(grafo_carro, P, destino_B)

        caminho_carro, custo_carro, nos_exp = resultado
        buscas_realizadas += 1
        total_nos_exp += nos_exp

        if caminho_carro is None:
            continue

        if custo_carro < melhor_custo:
            melhor_custo    = custo_carro
            melhor_P        = P
            melhor_caminho  = caminho_carro
            melhor_dist_cam = dist_cam

    nos_exp_medio = total_nos_exp // buscas_realizadas if buscas_realizadas > 0 else 0
    return melhor_P, melhor_dist_cam, melhor_custo, melhor_caminho, nos_exp_medio


def plotar_mapa_ridesmart(G_osm, coord_A, no_P, caminho_carro,
                           coord_B, dist_cam, custo, nome_algoritmo,
                           otimizar_por='tempo'):
    """Plota o mapa do RideSmart com a rota multimodal."""
    coord_P = (G_osm.nodes[no_P]['y'], G_osm.nodes[no_P]['x'])
    coords_carro = [(G_osm.nodes[no]['y'], G_osm.nodes[no]['x']) for no in caminho_carro]

    mapa = folium.Map(location=[coord_A[0], coord_A[1]], zoom_start=14, tiles='cartodbpositron')

    if otimizar_por == 'tempo':
        custo_str = f"{custo/60:.1f} min de carro"
        cor_rota  = 'red'
        icone_cor = 'red'
    else:
        custo_str = f"{custo/1000:.2f} km de carro"
        cor_rota  = 'blue'
        icone_cor = 'blue'

    folium.Marker(
        [coord_A[0], coord_A[1]],
        tooltip="Origem (A)",
        popup=f"<b>Origem</b><br>Caminhe {dist_cam:.0f} m até o ponto verde.",
        icon=folium.Icon(color="darkblue", icon="home", prefix="fa")
    ).add_to(mapa)

    folium.Marker(
        coord_P,
        tooltip="Embarque (P)",
        popup=f"<b>Ponto de Embarque</b><br>{nome_algoritmo}<br>{custo_str}",
        icon=folium.Icon(color="green", icon="car", prefix="fa")
    ).add_to(mapa)

    folium.Marker(
        coords_carro[-1],
        tooltip="Destino (B)",
        popup="<b>Destino Final</b>",
        icon=folium.Icon(color=icone_cor, icon="flag", prefix="fa")
    ).add_to(mapa)

    # Trecho a pé: linha tracejada preta
    folium.PolyLine(
        locations=[[coord_A[0], coord_A[1]], coord_P],
        color='black', weight=3, opacity=0.7, dash_array='6 6',
        tooltip=f"Caminhada: {dist_cam:.0f} m"
    ).add_to(mapa)

    # Trecho de carro
    folium.PolyLine(
        locations=coords_carro,
        color=cor_rota, weight=5, opacity=0.85,
        tooltip=f"Carro ({nome_algoritmo}): {custo_str}"
    ).add_to(mapa)

    return mapa

print("✅ Funções selecionar_melhor_embarque e plotar_mapa_ridesmart definidas.")


✅ Funções selecionar_melhor_embarque e plotar_mapa_ridesmart definidas.


## RideSmart — Dijkstra Simples


In [10]:
# ============================================================
# RIDESMART — DIJKSTRA SIMPLES
# ============================================================

NOME_ALGORITMO = "Dijkstra Simples"

print("=" * 60)
print(f"  🔍 {NOME_ALGORITMO}")
print(f"  Origem : {endereco_origem}")
print(f"  Destino: {endereco_destino}")
print(f"  Raio   : {raio_caminhada:.0f} m")
print("=" * 60)

# ── 1. MELHOR ROTA POR DISTÂNCIA ──────────────────────────────
print("\n[Distância] Buscando melhor ponto de embarque...")
P_d, cam_d, custo_d, rota_d, nexp_d = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_dist,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_simples,
    coordenadas      = coordenadas,
    otimizar_por     = 'distancia'
)

if P_d and rota_d:
    print(f"  ✅ Melhor embarque : nó {P_d}")
    print(f"     Caminhada       : {cam_d:.0f} m")
    print(f"     Distância carro : {custo_d/1000:.2f} km")
    print(f"     Nós expandidos  : {nexp_d}")
    mapa_dist = plotar_mapa_ridesmart(
        G_osm, coord_A, P_d, rota_d, coord_B,
        cam_d, custo_d, f"{NOME_ALGORITMO} — Distância", otimizar_por='distancia'
    )
    print("\n📍 Mapa — Melhor rota por DISTÂNCIA:")
    display(mapa_dist)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

# ── 2. MELHOR ROTA POR TEMPO (TRÂNSITO SINTÉTICO) ─────────────
print("\n[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...")
P_t, cam_t, custo_t, rota_t, nexp_t = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_simples,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_t and rota_t:
    print(f"  ✅ Melhor embarque : nó {P_t}")
    print(f"     Caminhada       : {cam_t:.0f} m")
    print(f"     Tempo carro     : {custo_t/60:.1f} min (com trânsito sintético)")
    print(f"     Nós expandidos  : {nexp_t}")
    mapa_tempo = plotar_mapa_ridesmart(
        G_osm, coord_A, P_t, rota_t, coord_B,
        cam_t, custo_t, f"{NOME_ALGORITMO} — Tempo", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (trânsito sintético):")
    display(mapa_tempo)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")


# ── 3. MELHOR ROTA POR TEMPO (SEM TRÂNSITO) ─────────────
print("\n[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...")
P_tl, cam_tl, custo_tl, rota_tl, nexp_tl = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo_livre,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_simples,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_tl and rota_tl:
    print(f"  ✅ Melhor embarque : nó {P_tl}")
    print(f"     Caminhada       : {cam_tl:.0f} m")
    print(f"     Tempo carro     : {custo_tl/60:.1f} min (SEM trânsito)")
    print(f"     Nós expandidos  : {nexp_tl}")
    mapa_tempo_livre = plotar_mapa_ridesmart(
        G_osm, coord_A, P_tl, rota_tl, coord_B,
        cam_tl, custo_tl, f"{NOME_ALGORITMO} — Tempo Livre", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (Sem trânsito):")
    display(mapa_tempo_livre)


  🔍 Dijkstra Simples
  Origem : Midway Mall, Natal, Brazil
  Destino: Ponta Negra, Natal, Brazil
  Raio   : 150 m

[Distância] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Distância carro : 8.89 km
     Nós expandidos  : 10058

📍 Mapa — Melhor rota por DISTÂNCIA:



[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 43.5 min (com trânsito sintético)
     Nós expandidos  : 9752

📍 Mapa — Melhor rota por TEMPO (trânsito sintético):



[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 8.9 min (SEM trânsito)
     Nós expandidos  : 9261

📍 Mapa — Melhor rota por TEMPO (Sem trânsito):


## RideSmart — Dijkstra com Heap


In [11]:
# ============================================================
# RIDESMART — DIJKSTRA COM HEAP
# ============================================================

NOME_ALGORITMO = "Dijkstra com Heap"

print("=" * 60)
print(f"  🔍 {NOME_ALGORITMO}")
print(f"  Origem : {endereco_origem}")
print(f"  Destino: {endereco_destino}")
print(f"  Raio   : {raio_caminhada:.0f} m")
print("=" * 60)

# ── 1. MELHOR ROTA POR DISTÂNCIA ──────────────────────────────
print("\n[Distância] Buscando melhor ponto de embarque...")
P_d, cam_d, custo_d, rota_d, nexp_d = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_dist,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_heap,
    coordenadas      = coordenadas,
    otimizar_por     = 'distancia'
)

if P_d and rota_d:
    print(f"  ✅ Melhor embarque : nó {P_d}")
    print(f"     Caminhada       : {cam_d:.0f} m")
    print(f"     Distância carro : {custo_d/1000:.2f} km")
    print(f"     Nós expandidos  : {nexp_d}")
    mapa_dist = plotar_mapa_ridesmart(
        G_osm, coord_A, P_d, rota_d, coord_B,
        cam_d, custo_d, f"{NOME_ALGORITMO} — Distância", otimizar_por='distancia'
    )
    print("\n📍 Mapa — Melhor rota por DISTÂNCIA:")
    display(mapa_dist)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

# ── 2. MELHOR ROTA POR TEMPO (TRÂNSITO SINTÉTICO) ─────────────
print("\n[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...")
P_t, cam_t, custo_t, rota_t, nexp_t = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_heap,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_t and rota_t:
    print(f"  ✅ Melhor embarque : nó {P_t}")
    print(f"     Caminhada       : {cam_t:.0f} m")
    print(f"     Tempo carro     : {custo_t/60:.1f} min (com trânsito sintético)")
    print(f"     Nós expandidos  : {nexp_t}")
    mapa_tempo = plotar_mapa_ridesmart(
        G_osm, coord_A, P_t, rota_t, coord_B,
        cam_t, custo_t, f"{NOME_ALGORITMO} — Tempo", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (trânsito sintético):")
    display(mapa_tempo)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")



  🔍 Dijkstra com Heap
  Origem : Midway Mall, Natal, Brazil
  Destino: Ponta Negra, Natal, Brazil
  Raio   : 150 m

[Distância] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Distância carro : 8.89 km
     Nós expandidos  : 10059

📍 Mapa — Melhor rota por DISTÂNCIA:



[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 43.5 min (com trânsito sintético)
     Nós expandidos  : 9753

📍 Mapa — Melhor rota por TEMPO (trânsito sintético):


## RideSmart — A*


In [12]:
# ============================================================
# RIDESMART — A*
# ============================================================

NOME_ALGORITMO = "A*"

print("=" * 60)
print(f"  🔍 {NOME_ALGORITMO}")
print(f"  Origem : {endereco_origem}")
print(f"  Destino: {endereco_destino}")
print(f"  Raio   : {raio_caminhada:.0f} m")
print("=" * 60)

# ── 1. MELHOR ROTA POR DISTÂNCIA ──────────────────────────────
print("\n[Distância] Buscando melhor ponto de embarque...")
P_d, cam_d, custo_d, rota_d, nexp_d = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_dist,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = a_estrela,
    coordenadas      = coordenadas,
    otimizar_por     = 'distancia'
)

if P_d and rota_d:
    print(f"  ✅ Melhor embarque : nó {P_d}")
    print(f"     Caminhada       : {cam_d:.0f} m")
    print(f"     Distância carro : {custo_d/1000:.2f} km")
    print(f"     Nós expandidos  : {nexp_d}")
    mapa_dist = plotar_mapa_ridesmart(
        G_osm, coord_A, P_d, rota_d, coord_B,
        cam_d, custo_d, f"{NOME_ALGORITMO} — Distância", otimizar_por='distancia'
    )
    print("\n📍 Mapa — Melhor rota por DISTÂNCIA:")
    display(mapa_dist)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

# ── 2. MELHOR ROTA POR TEMPO (TRÂNSITO SINTÉTICO) ─────────────
print("\n[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...")
P_t, cam_t, custo_t, rota_t, nexp_t = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = a_estrela,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_t and rota_t:
    print(f"  ✅ Melhor embarque : nó {P_t}")
    print(f"     Caminhada       : {cam_t:.0f} m")
    print(f"     Tempo carro     : {custo_t/60:.1f} min (com trânsito sintético)")
    print(f"     Nós expandidos  : {nexp_t}")
    mapa_tempo = plotar_mapa_ridesmart(
        G_osm, coord_A, P_t, rota_t, coord_B,
        cam_t, custo_t, f"{NOME_ALGORITMO} — Tempo", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (trânsito sintético):")
    display(mapa_tempo)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")
# ── 3. MELHOR ROTA POR TEMPO (SEM TRÂNSITO) ─────────────
print("\n[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...")
P_tl, cam_tl, custo_tl, rota_tl, nexp_tl = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo_livre,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = a_estrela,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_tl and rota_tl:
    print(f"  ✅ Melhor embarque : nó {P_tl}")
    print(f"     Caminhada       : {cam_tl:.0f} m")
    print(f"     Tempo carro     : {custo_tl/60:.1f} min (SEM trânsito)")
    print(f"     Nós expandidos  : {nexp_tl}")
    mapa_tempo_livre = plotar_mapa_ridesmart(
        G_osm, coord_A, P_tl, rota_tl, coord_B,
        cam_tl, custo_tl, f"{NOME_ALGORITMO} — Tempo Livre", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (Sem trânsito):")
    display(mapa_tempo_livre)



  🔍 A*
  Origem : Midway Mall, Natal, Brazil
  Destino: Ponta Negra, Natal, Brazil
  Raio   : 150 m

[Distância] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Distância carro : 8.89 km
     Nós expandidos  : 9461

📍 Mapa — Melhor rota por DISTÂNCIA:



[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 43.5 min (com trânsito sintético)
     Nós expandidos  : 7180

📍 Mapa — Melhor rota por TEMPO (trânsito sintético):



[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 8.9 min (SEM trânsito)
     Nós expandidos  : 223

📍 Mapa — Melhor rota por TEMPO (Sem trânsito):


## RideSmart — Dijkstra Bidirecional


In [16]:
# ============================================================
# RIDESMART — DIJKSTRA BIDIRECIONAL
# ============================================================

def dijkstra_bidir_wrapper(grafo_lista, origem, destino, **kw):
    """Adapta DijkstraBidirecional à assinatura (grafo, origem, destino) -> (caminho, custo, nos_exp)."""
    grafo_rev = {no: [] for no in grafo_lista}
    for u in grafo_lista:
        for v, w in grafo_lista[u]:
            grafo_rev[v].append((u, w))
    caminho, custo = DijkstraBidirecional(grafo_lista, grafo_rev, origem, destino)
    nos_exp = len(caminho) if caminho else 0
    return caminho, custo, nos_exp

grafo_dist_lst        = {no: list(grafo_dist[no].items())        for no in grafo_dist}
grafo_tempo_lst       = {no: list(grafo_tempo[no].items())       for no in grafo_tempo}
grafo_tempo_livre_lst = {no: list(grafo_tempo_livre[no].items()) for no in grafo_tempo_livre}

NOME_ALGORITMO = "Dijkstra Bidirecional"

print("=" * 60)
print(f"  🔍 {NOME_ALGORITMO}")
print(f"  Origem : {endereco_origem}")
print(f"  Destino: {endereco_destino}")
print(f"  Raio   : {raio_caminhada:.0f} m")
print("=" * 60)

# ── 1. MELHOR ROTA POR DISTÂNCIA ──────────────────────────────
print("\n[Distância] Buscando melhor ponto de embarque...")
P_d, cam_d, custo_d, rota_d, nexp_d = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_dist_lst,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_bidir_wrapper,
    coordenadas      = coordenadas,
    otimizar_por     = 'distancia'
)

if P_d and rota_d:
    print(f"  ✅ Melhor embarque : nó {P_d}")
    print(f"     Caminhada       : {cam_d:.0f} m")
    print(f"     Distância carro : {custo_d/1000:.2f} km")
    print(f"     Nós expandidos  : {nexp_d}")
    mapa_dist = plotar_mapa_ridesmart(
        G_osm, coord_A, P_d, rota_d, coord_B,
        cam_d, custo_d, f"{NOME_ALGORITMO} — Distância", otimizar_por='distancia'
    )
    print("\n📍 Mapa — Melhor rota por DISTÂNCIA:")
    display(mapa_dist)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

# ── 2. MELHOR ROTA POR TEMPO (TRÂNSITO SINTÉTICO) ─────────────
print("\n[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...")
P_t, cam_t, custo_t, rota_t, nexp_t = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo_lst,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_bidir_wrapper,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_t and rota_t:
    print(f"  ✅ Melhor embarque : nó {P_t}")
    print(f"     Caminhada       : {cam_t:.0f} m")
    print(f"     Tempo carro     : {custo_t/60:.1f} min (com trânsito sintético)")
    print(f"     Nós expandidos  : {nexp_t}")
    mapa_tempo = plotar_mapa_ridesmart(
        G_osm, coord_A, P_t, rota_t, coord_B,
        cam_t, custo_t, f"{NOME_ALGORITMO} — Tempo", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (trânsito sintético):")
    display(mapa_tempo)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

# ── 3. MELHOR ROTA POR TEMPO (SEM TRÂNSITO) ─────────────
print("\n[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...")
P_tl, cam_tl, custo_tl, rota_tl, nexp_tl = selecionar_melhor_embarque(
    G_osm            = G_osm,
    grafo_carro      = grafo_tempo_livre_lst,
    grafo_dist_dict  = grafo_dist,
    origem_A         = no_A,
    destino_B        = no_B,
    dist_max_caminhada = raio_caminhada,
    algoritmo_busca  = dijkstra_bidir_wrapper,
    coordenadas      = coordenadas,
    otimizar_por     = 'tempo'
)

if P_tl and rota_tl:
    print(f"  ✅ Melhor embarque : nó {P_tl}")
    print(f"     Caminhada       : {cam_tl:.0f} m")
    print(f"     Tempo carro     : {custo_tl/60:.1f} min (SEM trânsito)")
    print(f"     Nós expandidos  : {nexp_tl}")
    mapa_tempo_livre = plotar_mapa_ridesmart(
        G_osm, coord_A, P_tl, rota_tl, coord_B,
        cam_tl, custo_tl, f"{NOME_ALGORITMO} — Tempo Livre", otimizar_por='tempo'
    )
    print("\n📍 Mapa — Melhor rota por TEMPO (Sem trânsito):")
    display(mapa_tempo_livre)
else:
    print("  ⚠️  Nenhum ponto de embarque válido encontrado.")

  🔍 Dijkstra Bidirecional
  Origem : Midway Mall, Natal, Brazil
  Destino: Ponta Negra, Natal, Brazil
  Raio   : 150 m

[Distância] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Distância carro : 8.89 km
     Nós expandidos  : 87

📍 Mapa — Melhor rota por DISTÂNCIA:



[Tempo + Trânsito Sintético] Buscando melhor ponto de embarque...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 43.5 min (com trânsito sintético)
     Nós expandidos  : 82

📍 Mapa — Melhor rota por TEMPO (trânsito sintético):



[Tempo Livre] Buscando melhor ponto de embarque (Sem trânsito)...
  ✅ Melhor embarque : nó 505552550
     Caminhada       : 86 m
     Tempo carro     : 8.9 min (SEM trânsito)
     Nós expandidos  : 84

📍 Mapa — Melhor rota por TEMPO (Sem trânsito):


#Rota sem caminhada

In [17]:
# ---- CASO 4: SEM CAMINHADA (P = A)
print("=== CASO 4: Rota direta A → B (sem caminhada) ===")

# Dist
cam4_d, custo4_d, nexp4_d = dijkstra_heap(grafo_dist, no_A, no_B)
print(f"  Distância : {custo4_d/1000:.2f} km  |  Nós expandidos: {nexp4_d}")

# Tempo sem trânsito
cam4_tl, custo4_tl, nexp4_tl = dijkstra_heap(grafo_tempo_livre, no_A, no_B)
print(f"  Tempo livre: {custo4_tl/60:.1f} min  |  Nós expandidos: {nexp4_tl}")

# Tempo c/ trânsito sintético
cam4_t, custo4_t, nexp4_t = dijkstra_heap(grafo_tempo, no_A, no_B)
print(f"  Tempo trânsito: {custo4_t/60:.1f} min  |  Nós expandidos: {nexp4_t}")

# Mapa
mapa_caso4 = plotar_mapa_ridesmart(
    G_osm, coord_A, no_A, cam4_t, coord_B,
    0, custo4_t, "Caso 4 — Sem caminhada", otimizar_por='tempo'
)
display(mapa_caso4)

=== CASO 4: Rota direta A → B (sem caminhada) ===
  Distância : 8.97 km  |  Nós expandidos: 10106
  Tempo livre: 9.0 min  |  Nós expandidos: 9308
  Tempo trânsito: 44.4 min  |  Nós expandidos: 9948


#Ganho comparativo:

In [18]:
# ---------- CASO 5: GANHO AO CAMINHAR
print("=== CASO 5: Ganho obtido ao caminhar ===")

# Reuse os resultados já calculados nas seções anteriores.
# Exemplo com Dijkstra Heap, métrica de tempo com trânsito:
custo_sem_caminhada = custo4_t          # tempo direto A→B
custo_com_caminhada = custo_t           # melhor P encontrado pelo Heap
dist_caminhada      = cam_t             # metros caminhados até P

ganho_tempo = custo_sem_caminhada - custo_com_caminhada  # segundos

print(f"  Sem caminhada (A direto): {custo_sem_caminhada/60:.1f} min")
print(f"  Com caminhada ({dist_caminhada:.0f} m):   {custo_com_caminhada/60:.1f} min")

if ganho_tempo > 0:
    print(f"  ✅ Caminhar economizou {ganho_tempo/60:.1f} min")
elif ganho_tempo < 0:
    print(f"  ❌ Caminhar custou {-ganho_tempo/60:.1f} min a mais — não valeu a pena")
else:
    print(f"  ➡️  Caminhar não fez diferença")

ganho_dist = custo4_d - custo_d
print(f"\n  [Distância] Sem caminhada: {custo4_d/1000:.2f} km  →  Com caminhada: {custo_d/1000:.2f} km  (Δ {ganho_dist/1000:+.2f} km)")

=== CASO 5: Ganho obtido ao caminhar ===
  Sem caminhada (A direto): 44.4 min
  Com caminhada (86 m):   43.5 min
  ✅ Caminhar economizou 0.9 min

  [Distância] Sem caminhada: 8.97 km  →  Com caminhada: 8.89 km  (Δ +0.09 km)
